# Run Game

In [ ]:
from the_game_simulation import simple_game_strategy, GameOverError, shuffle_cards, draw_cards, initiate_game

In [36]:
def run_game(strategy, n_players: int = 3, n_shuffles: int = 200) -> dict:
    hand_size = 6 if n_players > 2 else 7
    shuffled_deck = shuffle_cards(n_shuffles=n_shuffles)   
    players, remaining_deck, stacks = initiate_game(n_players, shuffled_deck, hand_size)

    total_cards = lambda: len(remaining_deck) + sum(len(p) for p in players)
    turn = 0
    
    try:
        while total_cards() > 0:
            turn += 1
            #print(f"Turn {turn}: total_cards={total_cards()}, deck={len(remaining_deck)}, hands={[len(p) for p in players]}")
            if turn > 100:
                raise RuntimeError("Too many turns!")
            
            for i, player in enumerate(players):
                if len(player) == 0:
                    continue
                player, stacks = strategy(player, stacks, remaining_deck)
                player, remaining_deck = draw_cards(player, remaining_deck, hand_size)
                players[i] = player
        
        return {"victory": True, "stacks": {k: v.copy() for k, v in stacks.items()}, "cards_remaining": 0}
        
    except GameOverError:
        return {"victory": False, "stacks": {k: v.copy() for k, v in stacks.items()}, "cards_remaining": total_cards()}

In [37]:
# Run a game
result = run_game(simple_game_strategy, n_players=3, n_shuffles=200)
print(result)

{'victory': False, 'stacks': {'decreasing_stack_1': array([99, 96, 86, 83, 82, 75, 73, 69, 66, 57, 56, 50, 48, 47, 45, 41, 36,
        4,  3,  2]), 'decreasing_stack_2': array([99, 94, 81, 74, 58, 68, 78, 77, 67, 53, 42, 26, 10,  9,  8, 18, 28,
       24, 34, 16,  5]), 'increasing_stack_1': array([ 1,  7, 12, 17, 19, 22, 27, 30, 70, 76, 89, 90, 92, 97, 87, 91]), 'increasing_stack_2': array([ 1,  6, 13, 14, 15, 21, 29, 49, 61, 80, 84])}, 'cards_remaining': 33}


In [41]:
def run_simulation(strategy, n_games: int = 100, n_players: int = 3):
    """Run multiple games and collect data."""
    victories = []
    losses = []
    
    for _ in range(n_games):
        result = run_game(strategy, n_players=n_players)
        
        if result["victory"]:
            victories.append(result)
        else:
            losses.append(result)
    
    return {
        "victories": victories,
        "losses": losses,
        "win_rate": len(victories) / n_games
    }

# Run simulation
results = run_simulation(simple_game_strategy, n_games=10, n_players=3)

print(f"Win rate: {results['win_rate']*100:.1f}%")
print(f"Victories: {len(results['victories'])}")
print(f"Losses: {len(results['losses'])}")

# Analyze losses
if results['losses']:
    print("\n--- Sample loss ---")
    loss = results['losses'][0]
    print(f"Cards remaining: {loss['cards_remaining']}")
    print(f"Stack tops: {[v[-1] for v in loss['stacks'].values()]}")

Win rate: 0.5%
Victories: 54
Losses: 9946

--- Sample loss ---
Cards remaining: 25
Stack tops: [np.int64(2), np.int64(11), np.int64(97), np.int64(74)]
